# Data Cleaning & Structural Validation
## Customer Churn Sample Dataset

**Objective:** Transform the raw dataset into a reliable format for analysis using Python (Pandas), while checking missing values, duplicates, data types, categorical consistency, and structural validity.

## 1. Load the Dataset
The original CSV is loaded with Pandas.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('customer_churn_sample.csv')
print('Shape:', df.shape)
display(df.head())

## 2. Initial Inspection
We inspect missing values, duplicate rows, column names, and data types.

In [ ]:
print('Missing values by column:')
display(df.isna().sum())

print('Duplicate rows:', df.duplicated().sum())
print('\nData types:')
display(df.dtypes)

## 3. Cleaning Decisions
- **Missing values:** No missing values were found, so no imputation or row deletion was required.
- **Duplicates:** Duplicate records were checked and none were found.
- **Column headers:** Headers were stripped of whitespace and normalized to alphanumeric/underscore format.
- **Categorical strings:** Leading/trailing whitespace was removed.
- **Numeric fields:** Numeric columns were explicitly converted to numeric types.
- **CustomerID:** Kept as a string because it is an identifier, not a numeric measure.

In [ ]:
# Standardize headers
df.columns = (df.columns.str.strip()
              .str.replace(r'[^A-Za-z0-9]+', '_', regex=True)
              .str.strip('_'))

# Remove duplicate records
df = df.drop_duplicates().copy()

# Standardize categorical strings
categorical_cols = ['Gender', 'SubscriptionType', 'ContractType', 'PaymentMethod', 'Churn']
for col in categorical_cols:
    df[col] = df[col].astype('string').str.strip()

# Validate/convert numeric columns
for col in ['Age', 'TenureMonths', 'SupportTickets']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
for col in ['MonthlyCharges', 'TotalCharges']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['CustomerID'] = df['CustomerID'].astype('string')

## 4. Structural Validation
For this customer churn dataset, `TotalCharges` should equal `MonthlyCharges × TenureMonths`. We also check sensible non-negative ranges and valid churn labels.

In [ ]:
df['ExpectedTotalCharges'] = (df['MonthlyCharges'] * df['TenureMonths']).round(2)
df['ChargesCheck'] = np.isclose(df['TotalCharges'], df['ExpectedTotalCharges'], atol=0.01)

print('TotalCharges consistency:', df['ChargesCheck'].all())
print('Age > 0:', (df['Age'] > 0).all())
print('TenureMonths >= 0:', (df['TenureMonths'] >= 0).all())
print('MonthlyCharges >= 0:', (df['MonthlyCharges'] >= 0).all())
print('TotalCharges >= 0:', (df['TotalCharges'] >= 0).all())
print('SupportTickets >= 0:', (df['SupportTickets'] >= 0).all())
print('Churn values valid:', df['Churn'].isin(['Yes', 'No']).all())

## 5. Final Validation
After cleaning, confirm that the dataset contains no duplicates or missing values and review the final schema.

In [ ]:
# Remove validation helper columns before export
df = df.drop(columns=['ExpectedTotalCharges', 'ChargesCheck'])

print('Final shape:', df.shape)
print('Final missing values:', df.isna().sum().sum())
print('Final duplicate rows:', df.duplicated().sum())
display(df.dtypes)
display(df.head())

## 6. Export Clean Dataset
The cleaned dataset is exported as `customer_churn_cleaned.csv`.

In [ ]:
df.to_csv('customer_churn_cleaned.csv', index=False)
print('Cleaned dataset exported successfully.')

## Final Result
The dataset was successfully checked for missing values, duplicates, data-type issues, categorical formatting, and structural consistency. No records required deletion for missing data, and no duplicate records were present.